---

### ==================**`CMC 2-Year Follow-up Analysis`**===========================

In [1]:
#!/usr/bin/env python3
"""
CMC 1-Year Follow-up Analysis
Combines patient outcomes and tumor location analysis
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ============================================================================
# CONFIGURATION - Change these to match your data
# ============================================================================
INPUT_FILE = 'proceed_radiomics_103_patients(two_years).csv'
TUMOR_LOCATION_COLUMN = 'site'
OUTCOME_COLUMN = 'event_locoregional_recurrence'
LRR_VALUE = 1
NO_LRR_VALUE = 0

# ============================================================================
# LOAD AND ANALYZE DATA
# ============================================================================
df = pd.read_csv(INPUT_FILE)
total_patients = len(df)

# Patient outcomes
lrr_count = (df[OUTCOME_COLUMN] == LRR_VALUE).sum()
non_lrr_count = (df[OUTCOME_COLUMN] == NO_LRR_VALUE).sum()
lrr_pct = (lrr_count / total_patients) * 100
non_lrr_pct = (non_lrr_count / total_patients) * 100

# Tumor locations
location_counts = df[TUMOR_LOCATION_COLUMN].value_counts()
location_pct = (location_counts / total_patients) * 100

# LRR rate by location
lrr_by_location = {}
for location in location_counts.index:
    location_df = df[df[TUMOR_LOCATION_COLUMN] == location]
    lrr_in_location = (location_df[OUTCOME_COLUMN] == LRR_VALUE).sum()
    total_in_location = len(location_df)
    lrr_rate = (lrr_in_location / total_in_location) * 100
    lrr_by_location[location] = (lrr_rate, lrr_in_location, total_in_location)

# Print summary
print(f"\nTotal Patients: {total_patients}")
print(f"  LRR: {lrr_count} ({lrr_pct:.1f}%)")
print(f"  Non-LRR: {non_lrr_count} ({non_lrr_pct:.1f}%)")
print(f"\nTumor Sites: {len(location_counts)}")

# ============================================================================
# CREATE VISUALIZATIONS
# ============================================================================

# Figure 1: Patient Outcomes (Bar Chart)
fig1, ax1 = plt.subplots(figsize=(10, 6))

categories = ['LRR', 'Non-LRR']
counts = [lrr_count, non_lrr_count]
percentages = [lrr_pct, non_lrr_pct]
colors = ['#ff6b6b', '#51cf66']

bars = ax1.bar(categories, counts, color=colors, alpha=0.8, 
               edgecolor='black', linewidth=2)

for bar, count, pct in zip(bars, counts, percentages):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'n = {count}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=14, fontweight='bold')

ax1.set_ylabel('Number of Patients', fontsize=13, fontweight='bold')
ax1.set_title(f'Patient Outcomes (2-Year Follow-up)\nTotal N = {total_patients}', 
              fontsize=15, fontweight='bold', pad=20)
ax1.set_ylim(0, max(counts) * 1.25)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('two_year_follow-up_patient_outcomes.png', dpi=300, bbox_inches='tight')
plt.close()

# Figure 2: Tumor Location Distribution (Horizontal Bar)
fig2, ax2 = plt.subplots(figsize=(12, 8))

sorted_locs = location_counts.sort_values(ascending=True)
sorted_pct = location_pct[sorted_locs.index]

y_pos = np.arange(len(sorted_locs))
colors_rainbow = plt.cm.Set3(np.linspace(0, 1, len(sorted_locs)))

bars = ax2.barh(y_pos, sorted_locs.values, 
                color=colors_rainbow, edgecolor='black', linewidth=1.5)

for i, (count, pct) in enumerate(zip(sorted_locs.values, sorted_pct.values)):
    ax2.text(count + max(sorted_locs.values)*0.03, i,
             f'n={count} ({pct:.1f}%)',
             va='center', fontsize=10, fontweight='bold')

ax2.set_yticks(y_pos)
ax2.set_yticklabels(sorted_locs.index, fontsize=10)
ax2.set_xlabel('Number of Patients', fontsize=13, fontweight='bold')
ax2.set_title(f'Tumor Location Distribution\nTotal N = {total_patients}', 
              fontsize=15, fontweight='bold', pad=20)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('two_year_follow-up_tumor_locations.png', dpi=300, bbox_inches='tight')
plt.close()

# Figure 3: LRR Rate by Location
fig3, ax3 = plt.subplots(figsize=(12, 8))

# Sort by LRR rate
sorted_lrr = dict(sorted(lrr_by_location.items(), key=lambda x: x[1][0]))
locations = list(sorted_lrr.keys())
rates = [v[0] for v in sorted_lrr.values()]

y_pos = np.arange(len(locations))
colors_risk = ['#51cf66' if r < 15 else '#ffd93d' if r < 30 else '#ff6b6b' for r in rates]

bars = ax3.barh(y_pos, rates, color=colors_risk, edgecolor='black', linewidth=1.5)

for i, (location, (rate, lrr_n, total_n)) in enumerate(sorted_lrr.items()):
    ax3.text(rate + max(rates)*0.03, i,
             f'{rate:.1f}% ({lrr_n}/{total_n})',
             va='center', fontsize=10, fontweight='bold')

ax3.set_yticks(y_pos)
ax3.set_yticklabels(locations, fontsize=10)
ax3.set_xlabel('LRR Rate (%)', fontsize=13, fontweight='bold')
ax3.set_title(f'LRR Rate by Tumor Location\nTotal N = {total_patients}', 
              fontsize=15, fontweight='bold', pad=20)
ax3.grid(axis='x', alpha=0.3, linestyle='--')
ax3.set_xlim(0, max(rates) * 1.3)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#51cf66', edgecolor='black', label='Low (<15%)'),
    Patch(facecolor='#ffd93d', edgecolor='black', label='Medium (15-30%)'),
    Patch(facecolor='#ff6b6b', edgecolor='black', label='High (>30%)')
]
ax3.legend(handles=legend_elements, loc='lower right', title='LRR Risk')

plt.tight_layout()
plt.savefig('two_year_follow-up_lrr_by_location.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✓ Saved 3 figures:")
print(f"  1. two_year_follow-up_patient_outcomes.png")
print(f"  2. two_year_follow-up_tumor_locations.png")
print(f"  3. two_year_follow-up_lrr_by_location.png\n")


Total Patients: 102
  LRR: 44 (43.1%)
  Non-LRR: 58 (56.9%)

Tumor Sites: 4

✓ Saved 3 figures:
  1. two_year_follow-up_patient_outcomes.png
  2. two_year_follow-up_tumor_locations.png
  3. two_year_follow-up_lrr_by_location.png

